# Borealis-27b at full bf16 — NorSumm + NorQuAD

Runs [`NbAiLab/borealis-27b`](https://huggingface.co/NbAiLab/borealis-27b) at its **native bf16 precision — nothing quantized** — over **both** benchmarks:

| Benchmark | Task | Metric | Size |
|---|---|---|---|
| **NorSumm** | summarization | ROUGE-1/2/L | 33 articles |
| **NorQuAD** | extractive QA | Exact Match / F1 | 300 questions |

Borealis is currently missing from the QA table entirely, so this is what gets it there.

## Before you start

**Runtime → Change runtime type → `A100 GPU` + `High-RAM`.**

On **Colab Pro+**, also enable **background execution**. Both benchmarks off one model load takes several hours; without it, closing the browser kills the runtime.

The weights are **51.1 GiB**. Colab's A100 is 40 GB, so the model cannot be GPU-resident — accelerate splits it across GPU and CPU RAM. Everything stays bf16; the offloaded layers just make generation slow.

**The model is loaded once and reused for both benchmarks** — do not re-run the load cell between them.

Run the cells in order. Cell 1 checks your hardware **before** the 51 GiB download.

## 1. Check what card you were assigned

Run this first. If it says anything other than A100, use **Runtime → Disconnect and delete runtime** and try again — Colab assigns cards from a pool.

In [ ]:
import torch, psutil

WEIGHTS = 51.1  # GiB, bf16

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > A100 GPU + High-RAM.')

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram = psutil.virtual_memory().total / 1024**3
gpu_budget = vram - 6
spill = max(0.0, WEIGHTS - gpu_budget)

print(f'GPU: {name} ({vram:.1f} GiB)')
print(f'RAM: {ram:.1f} GiB')
print(f'Weights: {WEIGHTS} GiB bf16\n')

if vram < 30:
    print(f'STOP. {name} is too small. Disconnect and delete the runtime, then retry for an A100.')
elif spill == 0:
    print('Excellent — fully GPU-resident. Expect ~10-20 minutes for all 33 articles.')
elif spill > ram - 8:
    print(f'STOP. {spill:.1f} GiB must spill to RAM but only {ram-8:.1f} GiB is usable.')
    print('Switch to a High-RAM runtime.')
else:
    print(f'OK: {gpu_budget:.1f} GiB on GPU, {spill:.1f} GiB offloaded to RAM.')
    print('Expect ~1-3 tok/s, roughly 1.5-4 hours for all 33 articles.')
    if ram < 60:
        print('\nNOTE: this looks like standard-RAM. High-RAM is strongly recommended.')

## 2. Install dependencies

`bitsandbytes` is deliberately **not** installed — this run is unquantized.

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub pandas pyarrow
import transformers, accelerate
print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)

## 3. Mount Drive (recommended)

The script checkpoints after **every article**. Writing those checkpoints to Drive means a disconnect costs one article instead of the whole run. Skip this cell and it falls back to local storage, which is wiped when the runtime dies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/borealis_bench

## 4. Get the benchmark code

Public repo, so no token needed.

In [ ]:
!git clone -q --branch claude/norquad-norsumm-benchmarks-9qhyke \
    https://github.com/rymarinelli/embedding.git /content/embedding
%cd /content/embedding/benchmarks/scripts
!ls generate_summaries_borealis_bf16_colab.py

## 5. (Optional) Hugging Face login

`NbAiLab/borealis-27b` is not gated, so this is only useful for download rate limits. Skip it if you like.

In [ ]:
# from huggingface_hub import login
# login(token='hf_...')   # never commit this

## 6. Run

Downloads ~51 GiB, then generates. Watch the **first article's timing**: if it takes more than ~8 minutes, the offload is thrashing and the run won't finish in a sitting — stop and confirm you got High-RAM.

`REPETITION_PENALTY` is left at **1.0**, matching the original quantized run, so this isolates the effect of precision alone. Only change it for a separate second run.

In [ ]:
import generate_summaries_borealis_bf16_colab as run

# Load once, keep the handles — the QA benchmark below reuses them.
run.preflight()
processor, model = run.load_model()

print('\nrepetition_penalty =', run.REPETITION_PENALTY, '| temperature =', run.TEMPERATURE)
print('checkpointing to  =', run.OUT_PATH)

## 7. Benchmark A — NorSumm (summarization)

33 articles, ROUGE-1/2/L. Watch the **first article's timing**: more than ~8 minutes means the offload is thrashing — stop and confirm you got High-RAM.

Checkpoints after every article, so a disconnect costs one article. Re-run this cell to resume.

In [ ]:
run.main(model=model, processor=processor)

## 8. Benchmark B — NorQuAD (extractive QA)

300 questions, Exact Match / token-F1 — the numbers in Table 2. **Reuses the model already loaded above**, so there is no second 51 GiB download.

Decoding is greedy with `max_new_tokens=64`, matching `generate_qa_answers_local_gguf.py` so the EM/F1 are comparable with the API models. Note this differs from the summarization run, which matched its own baseline at temperature 0.2.

300 short answers on offloaded weights still takes a while, but each is far shorter than a summary. Checkpoints every question.

In [ ]:
import generate_qa_answers_borealis_bf16_colab as qa

qa.main(model=model, processor=processor)

## 9. Download both result files

| File | Goes in | Then run |
|---|---|---|
| `borealis-27b-bf16-full.json` (summaries) | `benchmarks/results/generated_summaries/` | `score_summaries.py` |
| `borealis-27b-bf16-full.json` (QA) | `benchmarks/results/qa_answers/` | `score_qa.py` |

Same filename, different directories — that is deliberate: both scorers take the model label from the filename stem, so this lands as `borealis-27b-bf16-full` in both tables.

In [ ]:
from google.colab import files
import os

# Each benchmark wrote to its own subdirectory under the same filename.
base = '/content/drive/MyDrive/borealis_bench'
for task in ('norsumm', 'qa'):
    f = os.path.join(base, task, 'borealis-27b-bf16-full.json')
    if os.path.exists(f):
        n = len(__import__('json').load(open(f)))
        print(f'{task}: {n} records -> downloading')
        files.download(f)
    else:
        print(f'{task}: not found — finish that benchmark first')